In [1]:
from __future__ import annotations

import os
import json
from pathlib import Path
from getpass import getpass

import weaviate
import weaviate.classes as wvc

from weaviate.classes.init import Auth
from weaviate.classes.query import MetadataQuery, Filter

In [2]:
if "WEAVIATE_URL" not in os.environ:
    os.environ["WEAVIATE_URL"] = getpass("WEAVIATE_URL: ")

if "WEAVIATE_API_KEY" not in os.environ:
    os.environ["WEAVIATE_API_KEY"] = getpass("WEAVIATE_API_KEY: ")

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass("OPENAI_API_KEY: ")

In [3]:
client = weaviate.connect_to_weaviate_cloud(
    cluster_url=os.environ["WEAVIATE_URL"],
    auth_credentials=Auth.api_key(os.environ["WEAVIATE_API_KEY"]),
    headers={
        "X-OpenAI-Api-Key": os.environ["OPENAI_API_KEY"],
    },
)

print("Client ready:", client.is_ready())

Client ready: True


In [4]:
VECTOR_DB_ROOT = Path(".").resolve()
NOTES_DIR = VECTOR_DB_ROOT / "notes"

print("VECTOR_DB_ROOT:", VECTOR_DB_ROOT)
print("NOTES_DIR:", NOTES_DIR)
print("Notes exists:", NOTES_DIR.exists())

VECTOR_DB_ROOT: /home/lipov/projects/Python_practice_sessions_05/vector_db
NOTES_DIR: /home/lipov/projects/Python_practice_sessions_05/vector_db/notes
Notes exists: True


In [5]:
ALLOWED_EXTENSIONS = {".ipynb", ".md"}

EXCLUDED_DIRS = {
    ".ipynb_checkpoints",
    "__pycache__",
}


def should_skip_path(path: Path) -> bool:
    return any(part in EXCLUDED_DIRS for part in path.parts)


def collect_weaviate_sources(root: Path) -> list[Path]:
    files = []

    for path in root.rglob("*"):
        if not path.is_file():
            continue

        if should_skip_path(path):
            continue

        if path.suffix.lower() not in ALLOWED_EXTENSIONS:
            continue

        if path.suffix.lower() == ".md" and "notes" not in path.parts:
            continue

        files.append(path)

    return sorted(files)

In [6]:
source_files = collect_weaviate_sources(VECTOR_DB_ROOT)

print("Files found:", len(source_files))

for path in source_files:
    print(path.relative_to(VECTOR_DB_ROOT))

Files found: 14
01_vectors.ipynb
02_txt_to_image.ipynb
03_crud.ipynb
04_hybrid_search.ipynb
05_generative_search.ipynb
06_veawiate_cloud.ipynb
07_w_cluster.ipynb
08_w_cluster_hybrid_search.ipynb
09_w_cluster_vectors.ipynb
10_w_cluster_txt_to_image.ipynb
11_w_cluster_generative_search.ipynb
12_w_cluster_rag_project_docs.ipynb
13_w_cluster_weaviate_focused_rag.ipynb
notes/weaviate_cloud_concepts.md


In [7]:
def read_markdown_file(path: Path) -> str:
    return path.read_text(encoding="utf-8")

In [8]:
def read_ipynb_file(path: Path) -> str:
    data = json.loads(path.read_text(encoding="utf-8"))

    parts = []

    for cell_index, cell in enumerate(data.get("cells", [])):
        cell_type = cell.get("cell_type", "")
        source = "".join(cell.get("source", []))

        if not source.strip():
            continue

        parts.append(
            f"[NOTEBOOK: {path.name}]\n"
            f"[CELL TYPE: {cell_type}]\n"
            f"[CELL INDEX: {cell_index}]\n\n"
            f"{source}"
        )

    return "\n\n---\n\n".join(parts)

In [9]:
def read_source_file(path: Path) -> str:
    suffix = path.suffix.lower()

    if suffix == ".ipynb":
        return read_ipynb_file(path)

    if suffix == ".md":
        return read_markdown_file(path)

    raise ValueError(f"Unsupported file type: {path}")

In [10]:
def chunk_text(text: str, *, chunk_size: int = 1800, overlap: int = 300) -> list[str]:
    text = text.strip()

    if not text:
        return []

    chunks = []
    start = 0

    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end].strip()

        if chunk:
            chunks.append(chunk)

        start += chunk_size - overlap

    return chunks

In [11]:
def get_source_kind(path: Path) -> str:
    if path.suffix.lower() == ".md":
        return "concept_note"

    if path.suffix.lower() == ".ipynb":
        return "notebook_implementation"

    return "unknown"


def build_weaviate_rag_records(files: list[Path], root: Path) -> list[dict]:
    records = []

    for path in files:
        text = read_source_file(path)
        chunks = chunk_text(text)

        relative_path = str(path.relative_to(root))
        source_kind = get_source_kind(path)

        for chunk_index, chunk in enumerate(chunks):
            records.append(
                {
                    "content": chunk,
                    "source": relative_path,
                    "file_name": path.name,
                    "file_type": path.suffix.lower().lstrip("."),
                    "source_kind": source_kind,
                    "chunk_index": chunk_index,
                }
            )

    return records

In [12]:
records = build_weaviate_rag_records(source_files, VECTOR_DB_ROOT)

print("Records:", len(records))

print(records[0]["source"])
print(records[0]["source_kind"])
print(records[0]["content"][:500])

Records: 80
01_vectors.ipynb
notebook_implementation
[NOTEBOOK: 01_vectors.ipynb]
[CELL TYPE: code]
[CELL INDEX: 0]

from __future__ import annotations
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

---

[NOTEBOOK: 01_vectors.ipynb]
[CELL TYPE: code]
[CELL INDEX: 1]

sns.set_theme(style='whitegrid', context='notebook')

---

[NOTEBOOK: 01_vectors.ipynb]
[CELL TYPE: code]
[CELL INDEX: 2]

color_1 = [40, 120, 60]
color_2 = [60, 50, 90]

---

[NOTEBOOK: 01_vectors.ipynb]
[CELL TYPE: code]
[CELL INDEX: 3]

fig = plt.figure(f


In [13]:
COLLECTION_NAME = "WeaviateFocusedRagChunk"

if client.collections.exists(COLLECTION_NAME):
    client.collections.delete(COLLECTION_NAME)

rag_chunks = client.collections.create(
    name=COLLECTION_NAME,
    vector_config=wvc.config.Configure.Vectors.text2vec_openai(
        model="text-embedding-3-small",
    ),
    generative_config=wvc.config.Configure.Generative.openai(
        model="gpt-4o-mini",
    ),
    properties=[
        wvc.config.Property(
            name="content",
            data_type=wvc.config.DataType.TEXT,
        ),
        wvc.config.Property(
            name="source",
            data_type=wvc.config.DataType.TEXT,
        ),
        wvc.config.Property(
            name="file_name",
            data_type=wvc.config.DataType.TEXT,
        ),
        wvc.config.Property(
            name="file_type",
            data_type=wvc.config.DataType.TEXT,
        ),
        wvc.config.Property(
            name="source_kind",
            data_type=wvc.config.DataType.TEXT,
        ),
        wvc.config.Property(
            name="chunk_index",
            data_type=wvc.config.DataType.INT,
        ),
    ],
)

print("Created collection:", COLLECTION_NAME)

Created collection: WeaviateFocusedRagChunk


In [14]:
result = rag_chunks.data.insert_many(records)

if result.errors:
    print("Import errors:")
    print(result.errors)
else:
    print("Import completed.")

Import completed.


In [15]:
count = rag_chunks.aggregate.over_all(total_count=True)

print("Objects in collection:", count.total_count)

Objects in collection: 80


In [16]:
response = rag_chunks.query.fetch_objects(
    limit=5,
    return_properties=[
        "source",
        "file_name",
        "file_type",
        "source_kind",
        "chunk_index",
        "content",
    ],
)

for obj in response.objects:
    print("Source:", obj.properties["source"])
    print("Kind:", obj.properties["source_kind"])
    print("Chunk:", obj.properties["chunk_index"])
    print(obj.properties["content"][:500])
    print("-" * 80)

Source: 09_w_cluster_vectors.ipynb
Kind: notebook_implementation
Chunk: 1
_1k():
    url = "https://raw.githubusercontent.com/weaviate-tutorials/intro-workshop/main/data/jeopardy_1k.json"
    return requests.get(url, timeout=30).json()

---

[NOTEBOOK: 09_w_cluster_vectors.ipynb]
[CELL TYPE: code]
[CELL INDEX: 5]

data = load_tiny_jeopardy()

print(type(data), len(data))
json_print(data[0])

---

[NOTEBOOK: 09_w_cluster_vectors.ipynb]
[CELL TYPE: code]
[CELL INDEX: 6]

def recreate_question_vector_collection(client, *, include_round=False):
    collection_name = "Ques
--------------------------------------------------------------------------------
Source: 13_w_cluster_weaviate_focused_rag.ipynb
Kind: notebook_implementation
Chunk: 2
nue

        parts.append(
            f"[NOTEBOOK: {path.name}]\n"
            f"[CELL TYPE: {cell_type}]\n"
            f"[CELL INDEX: {cell_index}]\n\n"
            f"{source}"
        )

    return "\n\n---\n\n".join(parts)

---

[NOTEBOOK: 13_w_cluster

In [17]:
query = "difference between BM25 and vector search"

response = rag_chunks.query.hybrid(
    query=query,
    alpha=0.4,
    limit=8,
    return_properties=[
        "source",
        "source_kind",
        "chunk_index",
        "content",
    ],
    return_metadata=MetadataQuery(score=True),
)

for obj in response.objects:
    print("Score:", obj.metadata.score)
    print("Source:", obj.properties["source"])
    print("Kind:", obj.properties["source_kind"])
    print("Chunk:", obj.properties["chunk_index"])
    print(obj.properties["content"][:700])
    print("-" * 80)

Score: 0.8952691555023193
Source: 12_w_cluster_rag_project_docs.ipynb
Kind: notebook_implementation
Chunk: 11
E: code]
[CELL INDEX: 40]

ask_notebooks("Explain what is the differnece between bm25 and vector search?")

---

[NOTEBOOK: 12_w_cluster_rag_project_docs.ipynb]
[CELL TYPE: code]
[CELL INDEX: 41]

client.close()
--------------------------------------------------------------------------------
Score: 0.7805297374725342
Source: notes/weaviate_cloud_concepts.md
Kind: concept_note
Chunk: 2
mpared with vectors stored in Weaviate.

Example:

```python
response = questions.query.near_text(
    query="animals",
    limit=5,
    return_metadata=MetadataQuery(distance=True),
)
```

A smaller distance usually means higher semantic similarity.

Vector search is useful when the user asks natural language questions or uses words different from the original document.

Example:

```text
query: hardware for machine learning
```

This can return documents about:

- GPU
- VRAM
- AI workloads
- par

In [18]:
query = "where is text-to-image search implemented with CLIP embeddings"

response = rag_chunks.query.hybrid(
    query=query,
    alpha=0.4,
    limit=8,
    return_properties=[
        "source",
        "source_kind",
        "chunk_index",
        "content",
    ],
    return_metadata=MetadataQuery(score=True),
)

for obj in response.objects:
    print("Score:", obj.metadata.score)
    print("Source:", obj.properties["source"])
    print("Kind:", obj.properties["source_kind"])
    print("Chunk:", obj.properties["chunk_index"])
    print(obj.properties["content"][:700])
    print("-" * 80)

Score: 0.8815768957138062
Source: 10_w_cluster_txt_to_image.ipynb
Kind: notebook_implementation
Chunk: 1
return [
        path
        for path in sorted(image_dir.iterdir())
        if path.suffix.lower() in {".jpg", ".jpeg", ".png"}
    ]


def embed_image(path: Path) -> list[float]:
    image = Image.open(path).convert("RGB")
    vector = clip_model.encode(
        image,
        normalize_embeddings=True,
        convert_to_numpy=True,
    )
    return vector.tolist()


def embed_text(text: str) -> list[float]:
    vector = clip_model.encode(
        text,
        normalize_embeddings=True,
        convert_to_numpy=True,
    )
    return vector.tolist()

---

[NOTEBOOK: 10_w_cluster_txt_to_image.ipynb]
[CELL TYPE: code]
[CELL INDEX: 6]

COLLECTION_NAME = "ClipImageCloud"

if client.collecti
--------------------------------------------------------------------------------
Score: 0.8106875419616699
Source: notes/weaviate_cloud_concepts.md
Kind: concept_note
Chunk: 4
ed with object pro

In [19]:
def ask_weaviate_rag(question: str, *, limit: int = 10, alpha: float = 0.4) -> None:
    response = rag_chunks.generate.hybrid(
        query=question,
        alpha=alpha,
        grouped_task=(
            "Answer the user's question using only the retrieved context. "
            "Use concept notes for definitions and explanations. "
            "Use notebook implementation chunks for code locations and implementation details. "
            "If the retrieved context contains both concept notes and notebooks, combine them. "
            "Keep the answer practical and concise. "
            "Mention relevant source files when useful. "
            "If the context is truly insufficient, say so explicitly."
        ),
        limit=limit,
        return_properties=[
            "source",
            "source_kind",
            "chunk_index",
            "content",
        ],
    )

    print("ANSWER:")
    print(response.generated)

    print("\nSOURCES:")
    for obj in response.objects:
        print(
            f"- {obj.properties['source']} "
            f"[{obj.properties['source_kind']}] "
            f"(chunk {obj.properties['chunk_index']})"
        )

In [20]:
ask_weaviate_rag(
    "Explain the difference between BM25 and vector search."
)

ANSWER:
The difference between BM25 and vector search lies primarily in their search methodologies:

### BM25 Keyword Search
- **Main Idea**: BM25 is a keyword-based search algorithm that matches exact or close word occurrences in text fields.
- **Good For**: It is best for scenarios when exact terms, names, IDs, categories, or technical keywords are crucial.
- **Example**:
  ```python
  response = questions.query.bm25(
      query="GPU",
      query_properties=["title", "content", "component"],
      limit=3,
  )
  ```
- **Behavior**: BM25 ranks documents higher if they contain the exact query term "GPU".

### Vector Search
- **Main Idea**: Vector search utilizes semantic similarity to find documents that are meaningfully relevant to the query, even if the exact terms are not present.
- **Good For**: It excels in situations where users pose natural language questions or employ synonyms and related concepts.
- **Example**:
  ```python
  response = questions.query.near_text(
      query

In [21]:
ask_weaviate_rag(
    "What is hybrid search and what does alpha mean?"
)

ANSWER:
### Hybrid Search Overview
Hybrid search combines two methodologies: BM25 (a keyword-based search algorithm) and vector search (which assesses semantic similarity). This combination leverages both keyword matching and semantic understanding to enhance search quality. 

### Code Implementation
In the context of Weaviate, hybrid search can be executed using the following code snippets:

#### Basic Hybrid Search
You can perform a hybrid search with a specified `alpha` (which balances keyword search and vector search):
```python
response = questions_hybrid.query.hybrid(
    query="animal",
    alpha=0.5,  # Adjusts the balance
    limit=3,
    return_properties=["question", "answer", "category"],
)
```
This example queries for "animal" and retrieves the top 3 results with a balanced approach between BM25 and vector search.

#### Comparison of Alpha Values
To see how different `alpha` values affect search results, you can iterate through a range of values:
```python
for alpha in [0.

In [22]:
ask_weaviate_rag(
    "Where is text-to-image search implemented and how does it work?"
)

ANSWER:
### Text-to-Image Search Implementation in Weaviate

Text-to-image search in Weaviate is implemented using CLIP embeddings. Here’s a concise overview of the process based on the provided context:

1. **Embedding Creation**: Both the local images and text queries are embedded into a shared CLIP vector space. You can utilize a local CLIP model to create image embeddings.

   ```python
   response = images.query.near_vector(
       near_vector=query_vector,
       target_vector="image_vector",
       limit=3,
   )
   ```

2. **Data Insertion**: Images are inserted into Weaviate with their associated vectors. You can use the following code snippet for inserting image data.

   ```python
   uuid = images.data.insert(
       properties={
           "filename": path.name,
           "local_path": str(path),
       },
       vector={
           "image_vector": vector,
       },
   )
   ```

3. **Querying**: For querying, you can leverage both text and image-based searches. For example,

In [23]:
ask_weaviate_rag(
    "How does Weaviate generative search work in these notebooks?"
)

ANSWER:
To manage and utilize Weaviate effectively, the following practical insights can be gathered from the provided context:

### Weaviate Cloud Concepts

**Connection:**  
In Weaviate Cloud, you connect to the cluster using the following code snippet:
```python
client = weaviate.connect_to_weaviate_cloud(
    cluster_url=os.environ["WEAVIATE_URL"],
    auth_credentials=Auth.api_key(os.environ["WEAVIATE_API_KEY"]),
    headers={
        "X-OpenAI-Api-Key": os.environ["OPENAI_API_KEY"],
    },
)
```
This requires the Weaviate URL, API key, and OpenAI API key configured in your environment.

**Basic Data Model:**
- **Collections** (similar to SQL tables)
- **Objects** (similar to SQL rows)
- **Properties** (similar to SQL columns)
- **UUID** (similar to primary keys)

### CRUD Operations
CRUD operations in Weaviate can be performed on collections. An example to retrieve data is:
```python
questions = client.collections.get("Question")
```
To create an object:
```python
uuid = question

In [24]:
ask_weaviate_rag(
    "How are CRUD operations implemented in the Weaviate Cloud notebooks?"
)

ANSWER:
To work with Weaviate Cloud, you will primarily focus on CRUD operations, embedding generation, and querying data effectively. Here’s a practical overview:

### Weaviate Cloud Connection
To connect to Weaviate Cloud, use the following code snippet:
```python
client = weaviate.connect_to_weaviate_cloud(
    cluster_url=os.environ["WEAVIATE_URL"],
    auth_credentials=Auth.api_key(os.environ["WEAVIATE_API_KEY"]),
    headers={
        "X-OpenAI-Api-Key": os.environ["OPENAI_API_KEY"],
    },
)
```
Make sure to set your environment variables for the `WEAVIATE_URL`, `WEAVIATE_API_KEY`, and `OPENAI_API_KEY`.

### Basic Data Model
Weaviate's structure can be likened to a SQL database:
- **Collection** = SQL Table
- **Object** = SQL Row
- **Property** = SQL Column
- **UUID** = SQL Primary Key

### CRUD Operations
Perform basic operations using the following examples:

- **Create an Object**:
```python
uuid = questions.data.insert({
    "question": "This planet is known as the Red Plane

In [25]:
def search_sources(question: str, *, limit: int = 10, alpha: float = 0.4) -> None:
    response = rag_chunks.query.hybrid(
        query=question,
        alpha=alpha,
        limit=limit,
        return_properties=[
            "source",
            "source_kind",
            "chunk_index",
            "content",
        ],
        return_metadata=MetadataQuery(score=True),
    )

    for obj in response.objects:
        print("Score:", obj.metadata.score)
        print("Source:", obj.properties["source"])
        print("Kind:", obj.properties["source_kind"])
        print("Chunk:", obj.properties["chunk_index"])
        print(obj.properties["content"][:700])
        print("-" * 80)

In [26]:
search_sources("BM25 vector search hybrid alpha")

Score: 0.9641997814178467
Source: 08_w_cluster_hybrid_search.ipynb
Kind: notebook_implementation
Chunk: 2
---

[NOTEBOOK: 08_w_cluster_hybrid_search.ipynb]
[CELL TYPE: code]
[CELL INDEX: 7]

response = questions_hybrid.query.fetch_objects(
    limit=5,
    return_properties=["question", "answer", "category"],
)

for obj in response.objects:
    print(obj.uuid)
    print(obj.properties)
    print("-" * 80)

---

[NOTEBOOK: 08_w_cluster_hybrid_search.ipynb]
[CELL TYPE: markdown]
[CELL INDEX: 8]

## Vector search -- semantic search

---

[NOTEBOOK: 08_w_cluster_hybrid_search.ipynb]
[CELL TYPE: code]
[CELL INDEX: 9]

print("Vector search")

response = questions_hybrid.query.near_text(
    query="animal",
    limit=3,
    return_properties=["question", "answer", "category"],
    return_metadata=Metad
--------------------------------------------------------------------------------
Score: 0.8999665975570679
Source: 04_hybrid_search.ipynb
Kind: notebook_implementation
Chunk: 2
return_propertie